# DriveSense-VLM — 00: Data Pipeline

**What this notebook does**
Runs the complete data pipeline from raw datasets to SFT-ready JSONL files:
1. nuScenes download + rarity filtering (6-signal composite score)
2. PySpark distributed scoring + analytics
3. DADA-2000 critical-moment extraction
4. Unified dataset builder (stratified train/val/test splits)
5. LLM counterfactual annotation (Anthropic API or `--mock-llm`)
6. SFT JSONL formatting

**GPU**: T4 is sufficient — this notebook is CPU/data-bound.  
**Estimated cost**: ~5 CU (T4, ~3 hours).  
**Persistent storage**: All outputs saved to Google Drive — safe to disconnect.

> **RECOVERY**: If Colab disconnects, rerun Cells 2–4 (setup) then continue
> from whichever section you were in — all intermediate files are on Drive.

In [ ]:
# Cell 2: Mount Google Drive + setup paths
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_ROOT = '/content/drive/MyDrive/DriveSense-VLM'
os.makedirs(PROJECT_ROOT, exist_ok=True)

# Clone repo to ephemeral storage (fast I/O for code)
!git clone https://github.com/jayanth922/DriveSense-VLM.git /content/DriveSense-VLM 2>/dev/null || (cd /content/DriveSense-VLM && git pull --quiet)
%cd /content/DriveSense-VLM

# Symlink data and outputs to Drive (persistent across disconnects)
!mkdir -p {PROJECT_ROOT}/data {PROJECT_ROOT}/outputs
!ln -sf {PROJECT_ROOT}/data /content/DriveSense-VLM/data 2>/dev/null || true
!ln -sf {PROJECT_ROOT}/outputs /content/DriveSense-VLM/outputs 2>/dev/null || true

print(f'Project root (Drive): {PROJECT_ROOT}')
print(f'Repo root (ephemeral): /content/DriveSense-VLM')

In [ ]:
# Cell 3: Install dependencies (must run every session — ephemeral environment)
# Upgrade setuptools first (Colab ships an outdated version that breaks PEP 660 editable installs)
!pip install --upgrade setuptools wheel build -q

# Pin numpy for nuScenes compatibility (devkit requires numpy<2.0)
!pip install "numpy>=1.24,<2.0" -q 2>/dev/null || true

# Project install — editable preferred, non-editable fallback
!pip install -e '.[data,eval]' -q 2>/dev/null || pip install '.[data,eval]' -q

# PySpark installed separately (not a project extra)
!pip install pyspark>=3.5 -q
print('Dependencies installed.')

In [ ]:
# Cell 4: Verify installation
import subprocess, sys

result = subprocess.run(
    [sys.executable, 'scripts/run_sanity_check.py'],
    capture_output=True, text=True
)
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])
    raise RuntimeError('Sanity check failed — see output above.')
print('\nAll imports verified ✓')

## Section 1 — nuScenes Download

We download the **nuScenes mini** split for development/testing (~1 GB).
For the full `trainval` dataset (~60 GB), upload it to your Google Drive
at `DriveSense-VLM/data/nuscenes/` and skip this cell.

nuScenes registration required: https://www.nuscenes.org/nuscenes#download

In [ ]:
# Cell 6: Download nuScenes mini (for testing)
import os
from pathlib import Path

NUSCENES_DIR = Path(PROJECT_ROOT) / 'data' / 'nuscenes'
NUSCENES_DIR.mkdir(parents=True, exist_ok=True)

if (NUSCENES_DIR / 'v1.0-mini').exists():
    print('nuScenes mini already present — skipping download.')
else:
    print('Installing nuScenes devkit ...')
    !pip install nuscenes-devkit -q

    # Option A: download via the official nuScenes script (requires registration)
    # Replace <YOUR_NUSCENES_TOKEN> with your download token from nuscenes.org
    # !python -m nuscenes.scripts.export_2d_annotations_as_json \
    #     --dataroot {NUSCENES_DIR} --version v1.0-mini

    # Option B: wget from a pre-signed URL (if you have one)
    # !wget -q -O /tmp/nuscenes_mini.tar.gz "<YOUR_PRESIGNED_URL>"
    # !tar -xzf /tmp/nuscenes_mini.tar.gz -C {NUSCENES_DIR}

    print('Download complete. Directory:', str(NUSCENES_DIR))

!ls {NUSCENES_DIR}

## Section 2 — nuScenes Rarity Filtering

In [ ]:
# Cell 8: Run nuScenes rarity filter
# Uses 6 binary signals (proximity, occlusion, density, adverse weather,
# VRU at intersection, cyclist). Keeps frames scoring >= 3.
!python scripts/run_nuscenes_filter.py \
    --nuscenes-root {PROJECT_ROOT}/data/nuscenes \
    --version v1.0-mini \
    --output-dir {PROJECT_ROOT}/outputs/data/nuscenes_filtered

import json
from pathlib import Path
filtered_dir = Path(PROJECT_ROOT) / 'outputs' / 'data' / 'nuscenes_filtered'
if filtered_dir.exists():
    files = list(filtered_dir.glob('*.json'))
    print(f'\nFiltered output: {len(files)} files in {filtered_dir}')

## Section 3 — PySpark Pipeline

Distributed rarity scoring with analytics (score distribution, signal
co-occurrence, temporal clustering). Uses the local Spark mode — no cluster needed.

In [ ]:
# Cell 10: Run PySpark rarity scoring pipeline
!python scripts/run_spark_pipeline.py \
    --version v1.0-mini \
    --nuscenes-root {PROJECT_ROOT}/data/nuscenes \
    --output-dir {PROJECT_ROOT}/outputs/data/spark_processed

from pathlib import Path
spark_dir = Path(PROJECT_ROOT) / 'outputs' / 'data' / 'spark_processed'
if spark_dir.exists():
    print('\nSpark output files:')
    for f in sorted(spark_dir.rglob('*'))[:20]:
        print(' ', f.relative_to(spark_dir))

## Section 4 — DADA-2000

**Manual upload required.** DADA-2000 is not publicly downloadable via script.

1. Download from the official source: https://github.com/JWFangit/LOTVS-DADA
2. Upload the extracted archive to Google Drive:
   `DriveSense-VLM/data/dada2000/DADA-2000/`
3. The expected structure is:
   ```
   data/dada2000/DADA-2000/<category>/<sequence>/images/
   data/dada2000/dada_text_annotations.xlsx
   ```
4. If DADA-2000 is not available, the cell below will skip gracefully.

In [ ]:
# Cell 12: Run DADA-2000 extraction (skip if data not present)
from pathlib import Path

dada_root = Path(PROJECT_ROOT) / 'data' / 'dada2000'
dada_data = dada_root / 'DADA-2000'

if dada_data.exists():
    print('DADA-2000 found — running extraction ...')
    !python scripts/run_dada_extraction.py \
        --dada-root {dada_root} \
        --output-dir {PROJECT_ROOT}/outputs/data/dada_extracted
    print('DADA-2000 extraction complete.')
else:
    print('DADA-2000 not found — skipping.')
    print(f'Expected at: {dada_data}')
    print('Upload manually to Google Drive and rerun this cell.')

## Section 5 — Build Unified Dataset

In [ ]:
# Cell 14: Build unified dataset (stratified train/val/test splits)
# UnifiedDatasetBuilder merges nuScenes + DADA-2000 and stratifies by source+category.
!python scripts/run_build_unified_dataset.py \
    --nuscenes-dir {PROJECT_ROOT}/outputs/data/nuscenes_filtered \
    --dada-dir {PROJECT_ROOT}/outputs/data/dada_extracted \
    --output-dir {PROJECT_ROOT}/outputs/data/unified

import json
from pathlib import Path

unified_dir = Path(PROJECT_ROOT) / 'outputs' / 'data' / 'unified'
for split in ('train', 'val', 'test'):
    p = unified_dir / f'{split}.jsonl'
    if p.exists():
        with open(p) as f:
            n = sum(1 for _ in f)
        print(f'  {split}: {n} examples')

## Section 6 — LLM Annotation

Generates structured JSON annotations using the Anthropic API.
~30% of frames with real hazards also get counterfactual augmentation.

- **Real run**: set `ANTHROPIC_API_KEY` below (costs ~\$5–10 for 800–1200 frames).
- **Mock run**: uses `--mock-llm` to generate placeholder annotations (no API key needed).

In [ ]:
# Cell 16: Set API key + run annotation pipeline
import os

# Uncomment and set your key for a real annotation run:
# os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...'

USE_MOCK = 'ANTHROPIC_API_KEY' not in os.environ
mock_flag = '--mock-llm' if USE_MOCK else ''
print(f'Running annotation pipeline (mock={USE_MOCK}) ...')

!python scripts/run_annotation_pipeline.py \
    --unified-dir {PROJECT_ROOT}/outputs/data/unified \
    --output-dir {PROJECT_ROOT}/outputs/data/annotated \
    --cache-dir {PROJECT_ROOT}/outputs/data/annotation_cache \
    {mock_flag}

print('Annotation pipeline complete.')

In [ ]:
# Cell 17: Format annotations into SFT JSONL (Qwen3-VL chat format)
!python scripts/run_annotation_pipeline.py \
    --format-only \
    --annotated-dir {PROJECT_ROOT}/outputs/data/annotated \
    --output-dir {PROJECT_ROOT}/outputs/data/sft_ready

print('SFT formatting complete.')

## Section 7 — Verification

In [ ]:
# Cell 19: Print dataset statistics (soft check — no hard assert)
import json
from pathlib import Path

sft_dir = Path(f'{PROJECT_ROOT}/outputs/data/sft_ready')

print('=' * 50)
print('  SFT Dataset Statistics')
print('=' * 50)

total = 0
for split in ('train', 'val', 'test'):
    p = sft_dir / f'sft_{split}.jsonl'
    if p.exists():
        with open(p) as f:
            lines_data = [json.loads(l) for l in f]
        n = len(lines_data)
        total += n
        print(f'\n  {split}: {n} examples  ({p.stat().st_size / 1024:.1f} KB)')
        if lines_data:
            first = lines_data[0]
            print(f'    Keys: {list(first.keys())}')
    else:
        print(f'\n  {split}: NOT YET CREATED')
        print(f'    Run the annotation pipeline (cells 15-17) first.')
        print(f'    Or use --mock-llm flag for testing.')

print('\n' + '=' * 50)
if not any((sft_dir / f'sft_{split}.jsonl').exists() for split in ('train', 'val', 'test')):
    print('  No SFT files found. This is expected if you:')
    print('     - Have not run the annotation pipeline yet')
    print('     - Used --mock-llm and no data was available')
    print('     - Need to run the full data pipeline first')
    print('  The training notebook (01) requires these files.')
else:
    print(f'  Total: {total} examples across all splits.')
    print('\nData pipeline complete! Proceed to 01_training.ipynb')